**Copyright**

Jelen forráskód a Budapesti Műszaki és Gazdaságtudományi Egyetemen tartott
"Deep Learning a gyakorlatban Python és LUA alapon" tantárgy segédanyagaként készült.

A tantárgy honlapja: http://smartlab.tmit.bme.hu/oktatas-deep-learning
Deep Learning kutatás: http://smartlab.tmit.bme.hu/deep-learning

A forráskódot GPLv3 licensz védi. Újrafelhasználás esetén lehetőség szerint kérjük
az alábbi szerzőt értesíteni.

2021 (c) Csapó Tamás Gábor (csapot kukac tmit pont bme pont hu),
Gyires-Tóth Bálint, Zainkó Csaba



Links:

[keras-tuner] https://github.com/keras-team/keras-tuner

[blog post] https://www.mikulskibartosz.name/using-keras-tuner-to-tune-hyperparameters-of-a-tensorflow-model/

In [ ]:
# a keras-tuner-t vizsgáljuk, ami 2019-ben vált elérhetővé
# (https://twitter.com/fchollet/status/1189992078991708160?lang=en),
# most is aktívan fejlesztik, mindenképp érdemes foglalkozni vele,
# mert nagy erőkkel dolgozik rajta a keras csapata
# (https://github.com/keras-team/keras-tuner/commits/master)

# néhány tutorial:
# 1) https://www.mikulskibartosz.name/using-keras-tuner-to-tune-hyperparameters-of-a-tensorflow-model/
# 2) https://www.mikulskibartosz.name/using-hyperband-for-tensorflow-hyperparameter-tuning-with-keras-tuner/
# 3) https://www.mikulskibartosz.name/how-to-automaticallyselect-the-hyperparameters-of-a-resnet-neural-network/

In [ ]:
# telepítsük a keras-tuner-t

!pip install keras-tuner

In [ ]:
# adatok betöltése ugyanúgy, mint eddig

import tensorflow as tf
from tensorflow import keras
import numpy as np

fashion_mnist = keras.datasets.fashion_mnist

(train_images, train_labels), (test_images, test_labels) = fashion_mnist.load_data()

train_images = train_images / 255.0
test_images = test_images / 255.0

train_images = train_images.reshape(len(train_images), 28, 28, 1)
test_images = test_images.reshape(len(test_images), 28, 28, 1)


In [ ]:
# először készítünk egy modell-generáló függvényt,
# ami a hp. hiperparaméterekből hálót állít elő
# a hyperas-hoz hasonlóan a keras-tuner-ben is különbözőképpen lehet megadni a
# hiperparaméter tartományokat:
# - hp.Int -> egész számok egy adott tartományban szétosztva, pl. konvolúciós filterek
# - hp.Choice -> választási lehetőség egy listából, pl. optimizáló
# - hp.Float -> az Int-hez hasonlóan kell min-max tartomány, pl. dropout-hoz jó
# - hp.Boolean -> bináris döntés, pl. egy adott háló-ág szerepeljen-e
# - hp.Fixed -> ha egy paramétert nem szeretnénk változtatni. erről majd később.

# most egy konvolúciós hálót rakunk össze a fashionmnist osztályozásra

def build_model(hp):
  model = keras.Sequential([
    keras.layers.Conv2D(
        filters=hp.Int('conv_1_filter', min_value=64, max_value=128, step=16),
        kernel_size=hp.Choice('conv_1_kernel', values = [3,5]),
        activation='relu',
        input_shape=(28,28,1)
    ),
    keras.layers.Conv2D(
        filters=hp.Int('conv_2_filter', min_value=32, max_value=64, step=16),
        kernel_size=hp.Choice('conv_2_kernel', values = [3,5]),
        activation='relu'
    ),
    keras.layers.Flatten(),
    keras.layers.Dense(
        units=hp.Int('dense_1_units', min_value=32, max_value=128, step=16),
        activation='relu'
    ),
    keras.layers.Dense(10, activation='softmax')
  ])


  # when using the 'sparse_categorical_crossentropy' loss, your targets should be integer targets.
  model.compile(optimizer=keras.optimizers.Adam(hp.Choice('learning_rate', values=[1e-2, 1e-3])),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

  return model



In [ ]:
# a legegyszerűbb hiperparaméter kereső algoritmus a randomsearch.
# a nevének megfelelően véletlenül választ a paraméterek közül
# most nem foglalkozunk vele, mert van ennél érdekesebb is

from kerastuner.tuners import RandomSearch

tuner = RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=10,
    directory='output',
    project_name='FashionMNIST_random')


In [ ]:
# a keras-tuner-ben egyelőre nincs implementálva a TPE
# helyette van GP, ami a kerastuner.tuners.bayesian-ban elérhető
# eredménye hasonló a TPE-hez, ezért nem foglalkozunk vele külön

In [ ]:
# helyette van Hyperband, ami először 'belenéz' többféle hálóba,
# és a rosszul teljesítő hálókat eldobja (pruning / metszés)
#
# azaz futtat pl. 2-2 epoch-ot különböző hiperparaméter kombinációk közül,
# (a lenti log-ban: initial_epoch: 0)
# aztán a jól teljesítők közül megint 2-2 epoch,
# (a lenti log-ban: initial_epoch: 2)
# és így tovább, iteratívan szűkíti a keresési teret
# (a lenti log-ban: initial_epoch: 4, ...)
# a vége felé pedig már végigmegy az összes epoch-on
#
# a 'factor' paraméterrel lehet szabályozni, hogy mennyire gyorsan szűkítsen,
# a 'max_epochs' pedig a nevének megfelelően max ennyi epoch-ot enged
#

from kerastuner.tuners import Hyperband

tuner = Hyperband(
    build_model,
    objective='val_accuracy',
    factor=3,
    max_epochs=10,
    directory='output',
    project_name='FashionMNIST_hyperband')

In [ ]:
# először nézzük meg, hogy mi lesz a keresési terünk

tuner.search_space_summary()

In [ ]:
# a tuner.search paraméterei a keras model.fit-hez hasonlóak
# ez végzi magát a hiperparaméter optimalizálást
# ha elindítjuk, kb. 15 percig fut

tuner.search(train_images, train_labels, epochs=10, validation_split=0.1)

In [ ]:
# utána a tuner-ből kinyerhetjük a legjobb modellt, és azt használhatjuk tovább,
# pl. újrataníthatunk vele

best_model = tuner.get_best_models(num_models=1)[0]
best_model.summary()

# best_model.fit(train_images, train_labels, epochs=20, validation_split=0.1, initial_epoch=4)

In [ ]:
# a tuner-ből kinyerhető, hogy mik voltak a legjobb hiperparaméterek

params_best = tuner.get_best_hyperparameters(num_trials=1)[0]
params_best.get_config()['values']

In [ ]:
# a tuner-ből kinyerhető, hogy mik lettek a legjobb eredmények
tuner.results_summary()

In [ ]:
# konklúzió: konvolúciós háló + keras-tuner + hyperband: megint sikerült picit jobb eredményt elérni

In [ ]:
# a keras-tuner-ben megtalálható néhány nevezetes hálónak a hiperoptolható változata,
# pl. Xception és ResNet

# itt most nem előretanított hálót töltünk be, mint a transfer learning esetén,
# hanem a háló szerkezetét, amiben néhány paramétert optimalizálhatunk
# a saját adataink függvényében

In [ ]:
from kerastuner.tuners import Hyperband
from kerastuner.applications import HyperResNet
from kerastuner import HyperParameters

hypermodel = HyperResNet(input_shape=(28, 28, 1), classes=10)

hp = HyperParameters()
hp.Choice('learning_rate', values=[1e-3, 1e-4])
hp.Fixed('optimizer', value='adam')

In [ ]:
tuner = Hyperband(
    hypermodel,
    objective='val_accuracy',
    hyperparameters=hp,
    tune_new_entries=False,
    max_epochs=5,
    directory='output',
    project_name='FashionMNIST_resnet')

In [ ]:
# nézzük meg, hogy mi lesz a keresési terünk

# mivel az optimizer-t fix-re állítottuk, azt nem fogja változtatni,
# és a 'tune_new_entries=False' miatt a többi hiperparamétert sem piszkálja
# csak a két learning_rate értéket fogja megnézni

tuner.search_space_summary()

In [ ]:
# ha beállítjuk, hogy 'tune_new_entries=False', akkor
# viszont a háló többi paraméterét is végignézné

tuner_large = Hyperband(
    hypermodel,
    objective='val_accuracy',
    hyperparameters=hp,
    # ez most True
    tune_new_entries=True,
    max_epochs=5,
    directory='output',
    project_name='FashionMNIST_resnet')

tuner_large.search_space_summary()

In [ ]:
# a ResNet-hez a címkéket onehot-enkódolni kell

train_labels_binary = keras.utils.to_categorical(train_labels)

In [ ]:
# itt most csak 2 tanítás megy végig a 2-féle learning rate-tel
# de mivel a ResNet hálózat nagy, ezért sokáig tart

tuner.search(train_images, train_labels_binary, validation_split=0.1)

In [ ]:
tuner.results_summary()

In [ ]:
params_best = tuner.get_best_hyperparameters(num_trials=1)[0]
params_best.get_config()['values']

In [ ]:
# a legjobb modellt visszaállíthatjuk a hypermodel és a params_best kombinációjából
model_best = tuner.hypermodel.build(params_best)

In [ ]:
print(model_best.summary())

In [ ]:
# itt a gyakorlat vége
# konklúzió a keras-tuner-ről: viszonylag új, aktívan fejlesztett rendszer